# Notebook 2 — Merge USA Medal Days with Market Data + Abnormal Returns (CAPM / Fama–French)

This notebook reads your medal-day panel (with `gold/silver/bronze/performance_score`) and builds a **finance-ready daily panel** that includes:

- **SPY** market return, volume change, and volatility proxies
- **Sector / sentiment-exposed tickers** (default: `NKE`, `XLY`, `DKNG`) returns + volume + volatility
- **Abnormal returns** for the tickers using:
  - **CAPM** (Market factor only) and/or
  - **Fama–French 3-factor** (MKT-RF, SMB, HML)

It outputs a single CSV panel you can use for regressions/event-study.

> Notes:
> - This notebook uses **online data sources** at runtime (Yahoo Finance + Ken French/FamaFrench via `pandas_datareader`). If your environment blocks web access, you can download the data manually and point the notebook to local CSVs.
> - Medal dates that fall on non-trading days are **shifted to the next NYSE trading day** (common in event studies).


In [ ]:
# Install dependencies if needed (comment out if already installed)
%pip -q install pandas numpy yfinance pandas_datareader pandas_market_calendars statsmodels tqdm

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import yfinance as yf
import pandas_market_calendars as mcal
from pandas_datareader import data as pdr
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

pd.set_option("display.max_columns", 200)

## 1) Load the medal-day dataset

This should be the output from Notebook 1 (USA-only), e.g.:
- `usa_medal_daily_counts_and_score.csv` or similar

Expected columns (minimum):
- `event_date_end` (date)
- `gold`, `silver`, `bronze`
- `performance_score`
- `total_medals`
- `any_medal_dummy` (optional; we can recreate)

If your file has `date` instead of `event_date_end`, the notebook will handle it.


In [ ]:
# ---- Input medal panel path (try common names) ----
CANDIDATE_MEDAL_FILES = [
    Path("usa_medal_daily_counts_and_score.csv"),
    Path("usa_medal_counts_and_score.csv"),
    Path("use_medal_counts_and_score.csv"),  # common typo
    Path("/mnt/data/usa_medal_daily_counts_and_score.csv"),
    Path("/mnt/data/usa_medal_counts_and_score.csv"),
    Path("/mnt/data/use_medal_counts_and_score.csv"),
]

medal_path = next((p for p in CANDIDATE_MEDAL_FILES if p.exists()), None)
if medal_path is None:
    raise FileNotFoundError(
        "Could not find medal-day CSV. Place it next to this notebook or set medal_path manually."
    )

print("Reading:", medal_path)
medals = pd.read_csv(medal_path)

print("Shape:", medals.shape)
display(medals.head())
print("Columns:", list(medals.columns))

In [ ]:
# Normalize date column name
if "event_date_end" in medals.columns:
    medals = medals.rename(columns={"event_date_end": "medal_date"})
elif "Date" in medals.columns:
    medals = medals.rename(columns={"Date": "medal_date"})
elif "date" in medals.columns:
    medals = medals.rename(columns={"date": "medal_date"})
else:
    raise ValueError("Could not find a date column. Expected 'event_date_end' or 'date'.")

medals["medal_date"] = pd.to_datetime(medals["medal_date"], errors="coerce")
medals = medals.dropna(subset=["medal_date"]).sort_values("medal_date").reset_index(drop=True)

# Ensure standard columns exist
for c in ["gold","silver","bronze"]:
    if c not in medals.columns:
        raise ValueError(f"Missing required column '{c}' in medal-day CSV.")

if "total_medals" not in medals.columns:
    medals["total_medals"] = medals[["gold","silver","bronze"]].sum(axis=1)

if "performance_score" not in medals.columns:
    medals["performance_score"] = 3*medals["gold"] + 2*medals["silver"] + 1*medals["bronze"]

if "any_medal_dummy" not in medals.columns:
    medals["any_medal_dummy"] = (medals["total_medals"] > 0).astype(int)

display(medals.head())
print("Medal date range:", medals["medal_date"].min().date(), "→", medals["medal_date"].max().date())

## 2) Align medal dates to NYSE trading days

Olympic events happen on weekends and holidays. For market merges, we map each medal date to the **next NYSE trading day** (if not already a trading day).

This yields: `trade_date`.


In [ ]:
nyse = mcal.get_calendar("NYSE")

start = medals["medal_date"].min() - pd.Timedelta(days=10)
end = medals["medal_date"].max() + pd.Timedelta(days=10)

schedule = nyse.schedule(start_date=start, end_date=end)
trading_days = pd.DatetimeIndex(schedule.index)

def to_next_trading_day(d: pd.Timestamp) -> pd.Timestamp:
    if d in trading_days:
        return d
    idx = trading_days.searchsorted(d)
    if idx >= len(trading_days):
        return pd.NaT
    return trading_days[idx]

medals["trade_date"] = medals["medal_date"].apply(to_next_trading_day)
medals = medals.dropna(subset=["trade_date"]).sort_values("trade_date").reset_index(drop=True)

display(medals.head(10)[["medal_date","trade_date","gold","silver","bronze","performance_score"]])
print("Unique trading days:", medals["trade_date"].nunique())

## 3) Download market data (Yahoo Finance)

Default tickers:
- `SPY` (market proxy)
- `NKE` (Nike)
- `XLY` (Consumer Discretionary ETF)
- `DKNG` (DraftKings)

You can edit `TICKERS`.


In [ ]:
TICKERS = ["SPY", "NKE", "XLY", "DKNG"]  # edit as desired

fetch_start = (medals["trade_date"].min() - pd.Timedelta(days=365)).date()
fetch_end = (medals["trade_date"].max() + pd.Timedelta(days=10)).date()

print("Fetching price data from", fetch_start, "to", fetch_end)
raw = yf.download(TICKERS, start=str(fetch_start), end=str(fetch_end), auto_adjust=False, progress=False)

# Extract fields
prices = raw["Adj Close"] if "Adj Close" in raw else raw["Close"]
close = raw["Close"]
volume = raw["Volume"]

# Ensure DataFrames (yfinance returns Series when one ticker)
if isinstance(prices, pd.Series): prices = prices.to_frame()
if isinstance(close, pd.Series): close = close.to_frame()
if isinstance(volume, pd.Series): volume = volume.to_frame()

display(prices.head())
display(volume.head())

In [ ]:
# Daily log returns
ret = np.log(prices).diff()

# Volume log change
log_volume_change = np.log(volume.replace(0, np.nan)).diff()

# Volatility proxies
abs_ret = ret.abs()              # close-to-close absolute return
vol_20d = ret.rolling(20).std()  # 20d rolling volatility

def to_long(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    out = df.stack().rename(value_name).reset_index()
    out.columns = ["date", "ticker", value_name]
    return out

market_long = (
    to_long(ret, "ret")
    .merge(to_long(log_volume_change, "log_volume_change"), on=["date","ticker"], how="left")
    .merge(to_long(abs_ret, "abs_ret"), on=["date","ticker"], how="left")
    .merge(to_long(vol_20d, "vol_20d"), on=["date","ticker"], how="left")
)

display(market_long.head(10))

## 4) Load daily Fama–French factors

We pull **daily** factors and RF:
- MKT-RF, SMB, HML, RF (all in decimals)

If this fails due to environment restrictions, download manually and load from CSV.


In [ ]:
ff = pdr.DataReader("F-F_Research_Data_Factors_daily", "famafrench")[0].copy()
ff.index = pd.to_datetime(ff.index)
ff = ff.rename(columns={"Mkt-RF":"MKT_RF"})
ff = ff[["MKT_RF","SMB","HML","RF"]].sort_index() / 100.0  # percent -> decimal

display(ff.head())
print("Factor date range:", ff.index.min().date(), "→", ff.index.max().date())

## 5) Abnormal returns (rolling CAPM + rolling FF3)

We compute abnormal returns as rolling-model residuals for each ticker **except SPY** (SPY is the market proxy).

Rolling window default: 252 trading days.


In [ ]:
ROLLING_WINDOW = 252  # change to 120 if you prefer

def compute_abnormal_returns_for_ticker(ticker: str, ret_df: pd.DataFrame, ff: pd.DataFrame, window: int = 252) -> pd.DataFrame:
    r = ret_df[ticker].dropna()
    dfm = pd.DataFrame({"ret": r}).join(ff, how="inner").dropna()
    dfm["excess_ret"] = dfm["ret"] - dfm["RF"]

    # CAPM
    X_capm = sm.add_constant(dfm[["MKT_RF"]])
    y = dfm["excess_ret"]
    capm_fit = RollingOLS(y, X_capm, window=window, min_nobs=max(60, int(window*0.5))).fit()
    capm_pred = (X_capm * capm_fit.params).sum(axis=1)
    dfm["abret_capm"] = y - capm_pred

    # FF3
    X_ff3 = sm.add_constant(dfm[["MKT_RF","SMB","HML"]])
    ff3_fit = RollingOLS(y, X_ff3, window=window, min_nobs=max(60, int(window*0.5))).fit()
    ff3_pred = (X_ff3 * ff3_fit.params).sum(axis=1)
    dfm["abret_ff3"] = y - ff3_pred

    out = dfm[["abret_capm","abret_ff3"]].copy()
    out["date"] = out.index
    out["ticker"] = ticker
    return out.reset_index(drop=True)

ab_list = []
for t in [x for x in TICKERS if x != "SPY"]:
    ab_list.append(compute_abnormal_returns_for_ticker(t, ret, ff, window=ROLLING_WINDOW))

abrets_long = pd.concat(ab_list, ignore_index=True)
display(abrets_long.head())
print(abrets_long.groupby("ticker").size())

In [ ]:
# Merge abnormal returns into market_long
market_long = market_long.merge(abrets_long, on=["date","ticker"], how="left")
display(market_long.head(10))

## 6) Build a wide market table for medal trading days and merge

We pivot ticker metrics into columns like:
- `ret_SPY`, `log_volume_change_SPY`, `vol_20d_SPY`
- `ret_NKE`, `abret_ff3_NKE`, etc.

Then merge onto medal-day panel by `trade_date`.


In [ ]:
# Restrict market_long to medal trading days
trade_dates = pd.DatetimeIndex(sorted(medals["trade_date"].unique()))
mkt_td = market_long[market_long["date"].isin(trade_dates)].copy()

value_cols = ["ret","log_volume_change","abs_ret","vol_20d","abret_capm","abret_ff3"]
wide = mkt_td.pivot(index="date", columns="ticker", values=value_cols)
wide.columns = [f"{v}_{t}" for v, t in wide.columns]
wide = wide.reset_index().rename(columns={"date":"trade_date"})

display(wide.head())

In [ ]:
final = medals.merge(wide, on="trade_date", how="left")

# Reorder key columns
front = ["trade_date","medal_date","gold","silver","bronze","total_medals","performance_score","any_medal_dummy"]
front = [c for c in front if c in final.columns]
final = final[front + [c for c in final.columns if c not in front]]

display(final.head(10))
print("Final panel shape:", final.shape)

# Quick missing checks for core series
for col in ["ret_SPY","log_volume_change_SPY","vol_20d_SPY"]:
    if col in final.columns:
        print(col, "missing:", final[col].isna().sum())

## 7) Export final panel CSV

This is your regression/event-study dataset.


In [ ]:
OUT_PANEL = Path("usa_medals_market_panel.csv")
final.to_csv(OUT_PANEL, index=False)
print("Wrote:", OUT_PANEL.resolve())

## 8) Optional: Export a compact regression subset

This creates a smaller CSV matching a typical regression schema.


In [ ]:
keep = [
    "trade_date","medal_date","gold","silver","bronze","performance_score","any_medal_dummy",
    "ret_SPY","log_volume_change_SPY","abs_ret_SPY","vol_20d_SPY",
    "ret_NKE","abret_capm_NKE","abret_ff3_NKE",
    "ret_XLY","abret_capm_XLY","abret_ff3_XLY",
    "ret_DKNG","abret_capm_DKNG","abret_ff3_DKNG",
]
keep = [c for c in keep if c in final.columns]

reg_df = final[keep].copy()
display(reg_df.head(10))

reg_df.to_csv("usa_medals_market_panel_regression_subset.csv", index=False)
print("Wrote: usa_medals_market_panel_regression_subset.csv")